# Chain with conditional chains

In [2]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
from rich import print 
from rich.logging import RichHandler
from rich.table import Table
from rich.console import Console
from rich.console import Console
import time
import os
import logging
import warnings

warnings.filterwarnings("ignore")

logger = logging.getLogger()
logger.setLevel(logging.INFO)

console_handler = RichHandler()
console_handler.setLevel(logging.INFO)

file_handler = logging.FileHandler("app.log", mode="a", encoding="utf-8")
file_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

console = Console()

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_key=AZURE_OPENAI_API_KEY,
    openai_api_version=AZURE_OPENAI_API_VERSION
)

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

# TASK - 1 [Prompt]
prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a movie review evaluator"),
    ("human", "Please categorize the movie review as positive or negative: {input}")
])

# TASK - 2 [LLM]
llm_structured_output = llm_openai.with_structured_output(llm_schema)

# TASK - 3 [Custom Runnable]
def pydantic_json(input: llm_schema) -> str:
    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

# Conditional Chain 1
# TASK - 1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")
])

# TASK - 2 [LLM]

# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser




# Conditional Chain 2 


In [3]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableBranch
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
from rich import print
from rich.logging import RichHandler
from rich.table import Table
from rich.console import Console
import time
import os
import logging
import warnings

warnings.filterwarnings("ignore")

logger = logging.getLogger()
logger.setLevel(logging.INFO)

console_handler = RichHandler()
console_handler.setLevel(logging.INFO)

file_handler = logging.FileHandler("app.log", mode="a", encoding="utf-8")
file_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

console = Console()

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_key=AZURE_OPENAI_API_KEY,
    openai_api_version=AZURE_OPENAI_API_VERSION
)

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

def insta_chain(text:dict):

    text = text="text"

    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system","You are a LinkedIn post generator"),
        ("human", "Create a post for the following text for Instagram: {text}")
    ])

    # TASK - 2 [LLM]
    
    # TASK - 3 [Str Parser]

    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

final_orchestrator.invoke({"input": "I loved this KFG movie"})


[04/08/26 14:16:18] INFO     HTTP Request: POST                                                     ]8;id=978818;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=368192;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 14:16:18] INFO     HTTP Request: POST                                                     ]8;id=461418;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=523327;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 14:16:18] INFO     HTTP Request: POST                                                     ]8;id=800216;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=60561;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 14:16:20] INFO     HTTP Request: POST                                                     ]8;id=601733;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=550962;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 14:16:20] INFO     HTTP Request: POST                                                     ]8;id=415129;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=686462;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/08/26 14:16:20] INFO     HTTP Request: POST                                                     ]8;id=440381;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=793791;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

'🌟 Embracing a Positive Mindset 🌟\n\nPositive thinking is more than just a buzzword—it’s the foundation for growth, resilience, and innovation in the workplace. When we shift our perspective and approach challenges with optimism, we open doors to new opportunities and inspire those around us.\n\nLet’s make positivity a daily habit! Celebrate small wins, encourage your teammates, and focus on solutions rather than obstacles. Together, we can build a culture where everyone thrives.\n\nHow do you cultivate positivity in your professional life? Share your tips below! 👇\n\n#PositiveMindset #WorkCulture #Growth #Leadership #Teamwork'